# MNIST LeNet 300-100 Pruning Notebook

This notebook demonstrates neural network pruning using MNIST dataset and LeNet with 300-100 architecture, including both dataset pruning and neural network pruning examples.

In [1]:
# Import Required Libraries
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from prune_neurals import Prunner

## Define LeNet with 300-100 Architecture

Create a LeNet model with the specified 300-100 fully connected layers.

In [2]:
class LeNet300_100(nn.Module):
    """LeNet 300-100: A simple fully-connected network for MNIST"""
    def __init__(self):
        super(LeNet300_100, self).__init__()
        # Fully connected layers only (no convolutions)
        self.fc1 = nn.Linear(28 * 28, 300)  # Input: 784 (28x28 flattened) -> 300
        self.fc2 = nn.Linear(300, 100)      # Hidden: 300 -> 100
        self.fc3 = nn.Linear(100, 10)       # Output: 100 -> 10 classes

    def forward(self, x):
        # Flatten the input image
        x = x.view(-1, 28 * 28)  # Flatten to [batch_size, 784]
        
        # Fully connected layers with ReLU activations
        x = torch.relu(self.fc1(x))  # 784 -> 300
        x = torch.relu(self.fc2(x))  # 300 -> 100
        x = self.fc3(x)              # 100 -> 10 (no activation, will use CrossEntropy)
        return x

# Create model instance
lenet = LeNet300_100()
print("Model architecture:")
print(lenet)
print(f"\nTotal parameters: {sum(p.numel() for p in lenet.parameters())}")
print("\nLayer details:")
print(f"FC1: {28*28} -> 300 = {(28*28)*300 + 300} parameters")
print(f"FC2: 300 -> 100 = {300*100 + 100} parameters") 
print(f"FC3: 100 -> 10 = {100*10 + 10} parameters")

Model architecture:
LeNet300_100(
  (fc1): Linear(in_features=784, out_features=300, bias=True)
  (fc2): Linear(in_features=300, out_features=100, bias=True)
  (fc3): Linear(in_features=100, out_features=10, bias=True)
)

Total parameters: 266610

Layer details:
FC1: 784 -> 300 = 235500 parameters
FC2: 300 -> 100 = 30100 parameters
FC3: 100 -> 10 = 1010 parameters


## Load and Preprocess MNIST Dataset

Load the MNIST dataset and create data loaders for training and testing.

In [3]:
# MNIST dataset transforms
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

# Load MNIST dataset
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=mnist_transform)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=mnist_transform)

# Create data loaders
batch_size = 100
mnist_train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True, num_workers=0)
mnist_test_loader = DataLoader(mnist_test, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Training samples: {len(mnist_train)}")
print(f"Test samples: {len(mnist_test)}")
print(f"Batch size: {batch_size}")
print(f"Training batches: {len(mnist_train_loader)}")
print(f"Test batches: {len(mnist_test_loader)}")

Training samples: 60000
Test samples: 10000
Batch size: 100
Training batches: 600
Test batches: 100


## Training and Testing Functions

Define functions for training and testing the model.

In [4]:
def train(model, train_loader, criterion, optimizer, device):
    """Training function"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    avg_loss = running_loss / total
    accuracy = 100. * correct / total
    print(f'Training: avg_loss = {avg_loss:.4f}, accuracy = {accuracy:.2f}%')
    return avg_loss, accuracy

def test(model, test_loader, criterion, device):
    """Testing function"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    avg_loss = running_loss / total
    accuracy = 100. * correct / total
    print(f'Testing:  avg_loss = {avg_loss:.4f}, accuracy = {accuracy:.2f}%')
    return avg_loss, accuracy

## Configure Model, Optimizer, and Loss Function

Set up the training configuration including device, optimizer, and loss function.

In [5]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Move model to device
lenet = lenet.to(device)

# Optimizer and loss function
optimizer = torch.optim.Adam(lenet.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print(f"Model moved to: {device}")
print(f"Optimizer: Adam with lr=0.001")
print(f"Loss function: CrossEntropyLoss")

Using device: cpu
Model moved to: cpu
Optimizer: Adam with lr=0.001
Loss function: CrossEntropyLoss


## Train the Model

Train the LeNet model for multiple epochs and monitor performance.

In [6]:
# Training loop
num_epochs = 10
train_accuracies = []
test_accuracies = []

print("Starting training...")
print("=" * 50)

for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    
    # Train and test
    train_loss, train_acc = train(lenet, mnist_train_loader, criterion, optimizer, device)
    test_loss, test_acc = test(lenet, mnist_test_loader, criterion, device)
    
    # Store accuracies
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    
    print("-" * 50)

print("Training completed!")
print(f"Final training accuracy: {train_accuracies[-1]:.2f}%")
print(f"Final test accuracy: {test_accuracies[-1]:.2f}%")

Starting training...
Epoch 1/10
Training: avg_loss = 0.2441, accuracy = 92.84%
Training: avg_loss = 0.2441, accuracy = 92.84%
Testing:  avg_loss = 0.1309, accuracy = 96.06%
--------------------------------------------------
Epoch 2/10
Testing:  avg_loss = 0.1309, accuracy = 96.06%
--------------------------------------------------
Epoch 2/10
Training: avg_loss = 0.0968, accuracy = 97.01%
Training: avg_loss = 0.0968, accuracy = 97.01%
Testing:  avg_loss = 0.0926, accuracy = 97.20%
--------------------------------------------------
Epoch 3/10
Testing:  avg_loss = 0.0926, accuracy = 97.20%
--------------------------------------------------
Epoch 3/10
Training: avg_loss = 0.0661, accuracy = 97.94%
Training: avg_loss = 0.0661, accuracy = 97.94%
Testing:  avg_loss = 0.0757, accuracy = 97.63%
--------------------------------------------------
Epoch 4/10
Testing:  avg_loss = 0.0757, accuracy = 97.63%
--------------------------------------------------
Epoch 4/10
Training: avg_loss = 0.0484, acc

## Save and Load Model State

Demonstrate saving the trained model and loading it back.

In [7]:
# Save the trained model
model_path = 'mnist_lenet_300_100.pth'
torch.save(lenet.state_dict(), model_path)
print(f"Model saved to: {model_path}")

# Test loading the model
lenet_loaded = LeNet300_100()
lenet_loaded.load_state_dict(torch.load(model_path, map_location=device))
lenet_loaded = lenet_loaded.to(device)
print("Model loaded successfully!")

# Verify the loaded model works
test_loss, test_acc = test(lenet_loaded, mnist_test_loader, criterion, device)
print(f"Loaded model test accuracy: {test_acc:.2f}%")

Model saved to: mnist_lenet_300_100.pth
Model loaded successfully!
Testing:  avg_loss = 0.0800, accuracy = 97.99%
Loaded model test accuracy: 97.99%
Testing:  avg_loss = 0.0800, accuracy = 97.99%
Loaded model test accuracy: 97.99%


## Neural Network Pruning Example

Now let's demonstrate neural network pruning on the trained LeNet model using the 300-100 fully connected layers.

In [10]:
# Initialize neural network pruner
neural_pruner = Prunner()

# Get the fully connected layers (fc1: 784->300, fc2: 300->100)
original_fc1 = lenet.fc1  # 784 -> 300
original_fc2 = lenet.fc2  # 300 -> 100

print("Original layer dimensions:")
print(f"FC1: {original_fc1.in_features} -> {original_fc1.out_features}")
print(f"FC2: {original_fc2.in_features} -> {original_fc2.out_features}")

# Test accuracy before pruning
print("\n=== Before Pruning ===")
test_loss, test_acc_before = test(lenet, mnist_test_loader, criterion, device)

Original layer dimensions:
FC1: 784 -> 300
FC2: 300 -> 100

=== Before Pruning ===
Testing:  avg_loss = 0.0800, accuracy = 97.99%
Testing:  avg_loss = 0.0800, accuracy = 97.99%


In [22]:
neural_pruner.method_name

{'Prune neurels': ['base', 'kmeans', 'distance-based-clustering', 'kmedoids'],
 'Prune dataset': ['base', 'kmeans', 'distance-based-clustering', 'kmedoids']}

In [69]:
# Create a fresh copy of the original model
lenet_pruned = LeNet300_100()
lenet_pruned.load_state_dict(torch.load(model_path, map_location=device))
lenet_pruned = lenet_pruned.to(device)


# Prune fc1 and fc2 layers
new_fc1, new_fc2 = neural_pruner.prune_neurals(
    layer1=lenet_pruned.fc1,
    layer2=lenet_pruned.fc2,
    prune_ratio=0.5,
    method='distance-based-clustering',
    device=device
)

# Replace the layers
lenet_pruned.fc1 = new_fc1
lenet_pruned.fc2 = new_fc2

# print(f"New layer dimensions:")
# print(f"FC1: {new_fc3.in_features} -> {new_fc1.out_features}")
# print(f"FC2: {new_fc2.in_features} -> {new_fc2.out_features}")

# Test the pruned model
test_loss, test_acc_after = test(lenet_pruned, mnist_test_loader, criterion, device)
    

            

Starting pruning layer 1 with prune_ratio=0.5...
Running CORESET to select 150 points from matrix of shape torch.Size([300, 100])...
starting reduce dimensions from 100 to 8
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([300, 8])...
262
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([262, 8])...
216
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([216, 8])...
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([300, 8])...
262
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([262, 8])...
216
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([216, 8])...
172
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([172, 8])...
124
CORESET completed, selected 150 points.
172
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shap

d:\lab\prune_neurals\prunner.py:50: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  u_tensor = torch.tensor(u, device=device, dtype=torch.float32)


Testing:  avg_loss = 3648.9161, accuracy = 87.76%


In [11]:
# Prune the network with different methods and ratios
prune_ratios = [0.3, 0.5, 0.7]
methods = ['base', 'kmeans', 'kmedoids', 'distance']

results = {}

for method in methods:
    print(f"\n=== Pruning with {method.upper()} method ===")
    method_results = {}
    
    for ratio in prune_ratios:
        print(f"\nPruning ratio: {ratio}")
        
        # Create a fresh copy of the original model
        lenet_pruned = LeNet300_100()
        lenet_pruned.load_state_dict(torch.load(model_path, map_location=device))
        lenet_pruned = lenet_pruned.to(device)
        
        try:
            # Prune fc1 and fc2 layers
            new_fc1, new_fc2 = neural_pruner.prune_neurals(
                layer1=lenet_pruned.fc1,
                layer2=lenet_pruned.fc2,
                prune_ratio=ratio,
                method=method,
                device=device
            )
            
            # Replace the layers
            lenet_pruned.fc1 = new_fc1
            lenet_pruned.fc2 = new_fc2
            
            print(f"New layer dimensions:")
            print(f"FC1: {new_fc1.in_features} -> {new_fc1.out_features}")
            print(f"FC2: {new_fc2.in_features} -> {new_fc2.out_features}")
            
            # Test the pruned model
            test_loss, test_acc_after = test(lenet_pruned, mnist_test_loader, criterion, device)
            
            method_results[ratio] = {
                'accuracy': test_acc_after,
                'fc1_out': new_fc1.out_features,
                'fc2_in': new_fc2.in_features
            }
            
            print(f"Accuracy drop: {test_acc_before - test_acc_after:.2f}%")
            
        except Exception as e:
            print(f"Error pruning with {method} at ratio {ratio}: {e}")
            method_results[ratio] = {'error': str(e)}
    
    results[method] = method_results


=== Pruning with BASE method ===

Pruning ratio: 0.3
Starting pruning layer 1 with prune_ratio=0.3...
Running CORESET to select 210 points from matrix of shape (300, 100)...
CORESET completed, selected 210 points.
New layer dimensions:
FC1: 784 -> 210
FC2: 210 -> 100
Testing:  avg_loss = 88661.8024, accuracy = 96.38%
Accuracy drop: 1.61%

Pruning ratio: 0.5
Starting pruning layer 1 with prune_ratio=0.5...
Running CORESET to select 150 points from matrix of shape (300, 100)...
CORESET completed, selected 150 points.
New layer dimensions:
FC1: 784 -> 150
FC2: 150 -> 100
Testing:  avg_loss = 88661.8024, accuracy = 96.38%
Accuracy drop: 1.61%

Pruning ratio: 0.5
Starting pruning layer 1 with prune_ratio=0.5...
Running CORESET to select 150 points from matrix of shape (300, 100)...
CORESET completed, selected 150 points.
New layer dimensions:
FC1: 784 -> 150
FC2: 150 -> 100
Testing:  avg_loss = 202898.2836, accuracy = 91.81%
Accuracy drop: 6.18%

Pruning ratio: 0.7
Starting pruning layer 1

d:\lab\prune_neurals\prunner.py:50: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  u_tensor = torch.tensor(u, device=device, dtype=torch.float32)


Testing:  avg_loss = 2104.8440, accuracy = 91.19%
Accuracy drop: 6.80%

Pruning ratio: 0.5
Starting pruning layer 1 with prune_ratio=0.5...
Running CORESET to select 150 points from matrix of shape torch.Size([300, 100])...
starting reduce dimensions from 100 to 8
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([300, 8])...
262
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([262, 8])...
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([300, 8])...
262
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([262, 8])...
216
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([216, 8])...
172
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([172, 8])...
124
CORESET completed, selected 150 points.
New layer dimensions:
FC1: 784 -> 150
FC2: 150 -> 100
216
Estimated number of clusters 

d:\lab\prune_neurals\prunner.py:50: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  u_tensor = torch.tensor(u, device=device, dtype=torch.float32)


Testing:  avg_loss = 2891.2974, accuracy = 88.31%
Accuracy drop: 9.68%

Pruning ratio: 0.7
Starting pruning layer 1 with prune_ratio=0.7...
Running CORESET to select 90 points from matrix of shape torch.Size([300, 100])...
starting reduce dimensions from 100 to 8
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([300, 8])...
262
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([262, 8])...
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([300, 8])...
262
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([262, 8])...
216
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([216, 8])...
172
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([172, 8])...
124
CORESET completed, selected 90 points.
New layer dimensions:
FC1: 784 -> 90
FC2: 90 -> 100
216
Estimated number of clusters (k):

...

Testing:  avg_loss = 1899.5431, accuracy = 91.85%
Accuracy drop: 6.14%

Pruning ratio: 0.5
Starting pruning layer 1 with prune_ratio=0.5...
Running CORESET to select 150 points from matrix of shape torch.Size([300, 100])...
starting reduce dimensions from 100 to 8
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([300, 8])...
262
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([262, 8])...
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([300, 8])...
262
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([262, 8])...
216
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([216, 8])...
172
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([172, 8])...
124
CORESET completed, selected 150 points.
New layer dimensions:
FC1: 784 -> 150
FC2: 150 -> 100
216
Estimated number of clusters 

d:\lab\prune_neurals\prunner.py:50: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  u_tensor = torch.tensor(u, device=device, dtype=torch.float32)


Testing:  avg_loss = 6525.0278, accuracy = 81.07%
Accuracy drop: 16.92%

Pruning ratio: 0.7
Starting pruning layer 1 with prune_ratio=0.7...
Running CORESET to select 90 points from matrix of shape torch.Size([300, 100])...
starting reduce dimensions from 100 to 8
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([300, 8])...
262
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([262, 8])...
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([300, 8])...
262
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([262, 8])...
216
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([216, 8])...
172
Estimated number of clusters (k): 1
Running l∞-CORESET on matrix of shape torch.Size([172, 8])...
124
CORESET completed, selected 90 points.
New layer dimensions:
FC1: 784 -> 90
FC2: 90 -> 100
216
Estimated number of clusters (k)

d:\lab\prune_neurals\prunner.py:50: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  u_tensor = torch.tensor(u, device=device, dtype=torch.float32)


Testing:  avg_loss = 11127.6216, accuracy = 73.19%
Accuracy drop: 24.80%


In [ ]:
# Display results summary
print("\n" + "="*60)
print("PRUNING RESULTS SUMMARY")
print("="*60)
print(f"Original accuracy: {test_acc_before:.2f}%")
print(f"Original FC layers: 300 -> 100")
print("\nResults by method and pruning ratio:")

for method, method_results in results.items():
    print(f"\n{method.upper()}:")
    for ratio, result in method_results.items():
        if 'error' in result:
            print(f"  Ratio {ratio}: ERROR - {result['error']}")
        else:
            acc = result['accuracy']
            fc1_out = result['fc1_out']
            fc2_in = result['fc2_in']
            acc_drop = test_acc_before - acc
            print(f"  Ratio {ratio}: {acc:.2f}% (drop: {acc_drop:.2f}%) | Layers: {fc1_out} -> {fc2_in}")

print("\n" + "="*60)

## Extract Feature Embeddings for Dataset Pruning

Extract feature embeddings from the trained model to demonstrate dataset pruning on real MNIST features.

In [ ]:
# Create a feature extractor (everything except the final classification layer)
feature_extractor = nn.Sequential(
    nn.Flatten(),    # Flatten input to [batch_size, 784]
    lenet.fc1,       # 784 -> 300
    nn.ReLU(),
    lenet.fc2,       # 300 -> 100
    nn.ReLU()
    # Exclude fc3 (final classification layer)
).to(device)

def extract_features(model, data_loader, device, max_samples=5000):
    """Extract feature embeddings from the model"""
    model.eval()
    features = []
    labels = []
    samples_processed = 0
    
    with torch.no_grad():
        for inputs, targets in data_loader:
            if samples_processed >= max_samples:
                break
                
            inputs = inputs.to(device)
            outputs = model(inputs)  # Shape: [batch_size, 100] (fc2 output)
            
            features.append(outputs.cpu())
            labels.append(targets)
            
            samples_processed += inputs.size(0)
    
    features_matrix = torch.cat(features, dim=0)[:max_samples]  # [N, 100]
    labels_tensor = torch.cat(labels, dim=0)[:max_samples]     # [N]
    
    return features_matrix, labels_tensor

# Extract features from training set
print("Extracting features from training set...")
train_features, train_labels = extract_features(feature_extractor, mnist_train_loader, device, max_samples=5000)
print(f"Extracted features shape: {train_features.shape}")
print(f"Feature dimension: {train_features.shape[1]} (fc2 output)")
print(f"Number of samples: {train_features.shape[0]}")

In [ ]:
# Apply dataset pruning on extracted features
print("\n=== Dataset Pruning on Real MNIST Features ===")

dataset_pruning_results = {}
prune_ratios_dataset = [0.3, 0.5, 0.7, 0.9]
methods_dataset = ['base', 'kmeans']  # Use fewer methods for faster execution

for method in methods_dataset:
    print(f"\nDataset pruning with {method.upper()} method:")
    method_results = {}
    
    for ratio in prune_ratios_dataset:
        try:
            # Prune the dataset
            coreset_indices = dataset_pruner.prune(
                train_features, 
                prune_ratio=ratio, 
                method=method, 
                device='cpu'  # Use CPU for dataset pruning to avoid memory issues
            )
            
            original_size = train_features.shape[0]
            pruned_size = len(coreset_indices)
            actual_ratio = 1 - (pruned_size / original_size)
            
            print(f"  Ratio {ratio}: {original_size} -> {pruned_size} samples "
                  f"(actual ratio: {actual_ratio:.3f})")
            
            method_results[ratio] = {
                'original_size': original_size,
                'pruned_size': pruned_size,
                'actual_ratio': actual_ratio,
                'indices': coreset_indices
            }
            
        except Exception as e:
            print(f"  Ratio {ratio}: ERROR - {e}")
            method_results[ratio] = {'error': str(e)}
    
    dataset_pruning_results[method] = method_results

print(f"\nDataset pruning completed!")
print(f"Original feature dimension: {train_features.shape[1]}")
print(f"Original dataset size: {train_features.shape[0]}")